# Exhaustive noisy-channel Bayesian transcripts

This notebook builds two deterministic question schedules and exhaustively enumerates all eight $K=3$ report histories for each. No latent value or channel coin is sampled or stored. `DeterministicDemoTokenizer` supplies reproducible pre-run construction fields only: `QwenRunner` re-applies the loaded Qwen tokenizer's chat template and persists the exact runtime prompt, input IDs, and token strings in each result row. The final cells demonstrate batched Qwen3.5 execution and tensor capture. Rerun the builder and execution cells after framework changes; saved cell output may reflect an earlier run.

In [2]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT))

In [3]:
from __future__ import annotations

import tempfile
from pathlib import Path

from mats_experiments.noisy_channel_bayesian import (
    CandidateEvidenceBayesianEnvironment,
    CandidateEvidenceDatasetGenerator,
    CandidateEvidenceQuestion,
    CaptureSpec,
    ExecutionConfig,
    MetricSpec,
    ModelConfig,
    NoisyChannelBayesianEnvironment,
    QwenRunner,
    RandomSubsetQuestion,
    SystemPrompt,
    TokenizerBinding,
    TranscriptDataset,
    TranscriptDatasetGenerator,
    XVsYPosteriorProbe,
    exact_pattern_mass,
    get_activation,
    get_answer_surface_logits,
    summarize_representation_control,
)


class DeterministicDemoTokenizer:
    """Offline construction tokenizer; QwenRunner reserializes with Qwen's tokenizer."""

    chat_template = "deterministic-demo-v1"
    name_or_path = "deterministic-demo"

    def apply_chat_template(self, messages, *, tokenize, add_generation_prompt, **_):
        text = "".join(f"<{m['role']}>{m['content']}" for m in messages)
        if add_generation_prompt:
            text += "<assistant>"
        return list(text.encode()) if tokenize else text

In [11]:
environment = NoisyChannelBayesianEnvironment(n=8, k=3, r_values="3/4")
probe = XVsYPosteriorProbe(
    x=2, y=7, reasoning_budget=40, allow_same=False, call_layout="conversation"
)
demo_tokenizer_binding = TokenizerBinding(DeterministicDemoTokenizer())
demo_system_prompt = SystemPrompt("Reason carefully from the stated noisy-channel model.")
builder = TranscriptDatasetGenerator(
    environment=environment,
    question=RandomSubsetQuestion(subset_size=4, replacement=False, sort=True),
    probe=probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
    seed=20260902,
)
dataset = builder.generate(num_question_sets=2)
assert len(dataset) == 2 * 2**3
dataset[:]

[{'schema_version': '1.0',
  'row_id': 'efee7fc4643ad49681e3d7df',
  'question_set_index': 0,
  'answer_pattern_index': 0,
  'answer_pattern': 'YYY',
  'domain': [1, 2, 3, 4, 5, 6, 7, 8],
  'n': 8,
  'k': 3,
  'reliabilities_exact': ['3/4', '3/4', '3/4'],
  'reliabilities': [0.75, 0.75, 0.75],
  'shared_reliability': True,
  'questions': [{'raw_draws': [5, 8, 6, 3], 'membership_set': [3, 5, 6, 8]},
   {'raw_draws': [1, 5, 8, 7], 'membership_set': [1, 5, 7, 8]},
   {'raw_draws': [6, 1, 5, 4], 'membership_set': [1, 4, 5, 6]}],
  'membership_sets': [[3, 5, 6, 8], [1, 5, 7, 8], [1, 4, 5, 6]],
  'observed_reports': ['YES', 'YES', 'YES'],
  'x': 2,
  'y': 7,
  'allow_same': False,
  'reasoning_budget': 40,
  'call_layout': 'conversation',
  'messages': [{'role': 'system',
    'content': 'Reason carefully from the stated noisy-channel model.'},
   {'role': 'user',
    'content': 'A value s is uniformly distributed over the integers 1 through 8.\nFor every question, the reported answer equals 

## Exact targets and mass normalization

The exact fields are rational strings. Floating-point mirrors are included for analysis and plotting.

In [12]:
first = dataset[0]
{
    "pattern": first["answer_pattern"],
    "evidence_mass": first["prior_predictive_exact"],
    "posterior": first["posterior_exact"],
    "x_vs_y": (first["x_posterior_exact"], first["y_posterior_exact"]),
    "target": first["ground_truth_choice"],
}

{'pattern': 'YYY',
 'evidence_mass': '1/8',
 'posterior': {'1': '9/64',
  '2': '1/64',
  '3': '3/64',
  '4': '3/64',
  '5': '27/64',
  '6': '9/64',
  '7': '3/64',
  '8': '9/64'},
 'x_vs_y': ('1/64', '3/64'),
 'target': 'Y'}

In [ ]:
masses = {index: exact_pattern_mass(dataset, index) for index in range(2)}
assert all(mass == 1 for mass in masses.values())
masses

## Uniform-history versus natural-distribution summaries

For illustration, the next cell attaches deterministic mock correctness values. Real result datasets get these fields from `QwenRunner`.

In [13]:
illustrative_rows = []
for row in dataset:
    illustrative_rows.append(
        {
            **row,
            "posterior_correct": (row["answer_pattern"] in {"YYY", "NNN"})
            if row["ground_truth_choice"] is not None
            else None,
        }
    )
TranscriptDataset(illustrative_rows).summarize()

{'row_count': 16,
 'defined_row_count': 16,
 'undefined_row_count': 0,
 'undefined_uniform_fraction': 0.0,
 'undefined_natural_mass': 0.0,
 'forced_tie_count': 4,
 'forced_tie_uniform_fraction': 0.25,
 'forced_tie_natural_mass': 0.25,
 'accuracy_eligible_count': 12,
 'accuracy_scored_count': 12,
 'uniform_defined_history_accuracy': 0.16666666666666666,
 'natural_distribution_accuracy': 0.125,
 'natural_distribution_question_set_count': 2,
 'uniform_history_parse_compliance': None,
 'natural_distribution_parse_compliance': None}

## Save/load and table inspection

In [14]:
demo_temporary_directory = tempfile.TemporaryDirectory(prefix="noisy_channel_bayesian_demo_")
experiment_dir = Path(demo_temporary_directory.name)
dataset.save(experiment_dir)
reloaded = TranscriptDataset.load(experiment_dir)
assert reloaded[0] == dataset[0]
reloaded.columns, reloaded.head(2), reloaded["answer_pattern"][:8]

(['schema_version',
  'row_id',
  'question_set_index',
  'answer_pattern_index',
  'answer_pattern',
  'domain',
  'n',
  'k',
  'reliabilities_exact',
  'reliabilities',
  'shared_reliability',
  'questions',
  'membership_sets',
  'observed_reports',
  'x',
  'y',
  'allow_same',
  'reasoning_budget',
  'call_layout',
  'messages',
  'serialized_prompt',
  'input_ids',
  'tokenizer_template_fingerprint',
  'prior_predictive_exact',
  'prior_predictive',
  'posterior_state',
  'posterior_exact',
  'posterior',
  'x_posterior_exact',
  'y_posterior_exact',
  'x_posterior',
  'y_posterior',
  'posterior_difference',
  'posterior_log_odds',
  'ground_truth_choice',
  'normative_comparison'],
 [{'schema_version': '1.0',
   'row_id': 'efee7fc4643ad49681e3d7df',
   'question_set_index': 0,
   'answer_pattern_index': 0,
   'answer_pattern': 'YYY',
   'domain': [1, 2, 3, 4, 5, 6, 7, 8],
   'n': 8,
   'k': 3,
   'reliabilities_exact': ['3/4', '3/4', '3/4'],
   'reliabilities': [0.75, 0.75, 0.

## Paired candidate-evidence control

The reduced environment derives per-candidate `AGREES`/`DISAGREES` evidence from each raw row. The exact posterior and natural-distribution weight remain identical. Raw sets and reports remain in `audit_metadata`, but the reduced renderer has no interface through which they can enter the prompt.

In [15]:
immediate_probe = XVsYPosteriorProbe(x=2, y=7, reasoning_budget=0, allow_same=False)
raw_immediate = TranscriptDatasetGenerator(
    environment=environment,
    question=RandomSubsetQuestion(subset_size=4, replacement=False, sort=True),
    probe=immediate_probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
    seed=20260902,
).generate(num_question_sets=2)
reduced_question = CandidateEvidenceQuestion()
reduced = CandidateEvidenceDatasetGenerator(
    environment=CandidateEvidenceBayesianEnvironment(n=8, k=3, r_values="3/4"),
    question=reduced_question,
    probe=immediate_probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
).generate(source_dataset=raw_immediate)
assert [row["source_row_id"] for row in reduced] == [row["row_id"] for row in raw_immediate]
assert all(
    raw["posterior_exact"] == compact["posterior_exact"]
    and raw["prior_predictive_exact"] == compact["prior_predictive_exact"]
    for raw, compact in zip(raw_immediate, reduced)
)
print(reduced[0]["messages"][-1]["content"])

A value s is uniformly distributed over the integers 1 through 8.
Candidate X means s=2 and candidate Y means s=7, so the candidates have equal prior probability.

For a candidate, AGREES means that the observed report matches the answer predicted by that candidate. A AGREES observation contributes its stated reliability to that candidate's likelihood. DISAGREES contributes one minus that reliability. Observations are conditionally independent.

Candidate X (s=2):
Observation 1: DISAGREES; reliability 0.75.
Observation 2: DISAGREES; reliability 0.75.
Observation 3: DISAGREES; reliability 0.75.

Candidate Y (s=7):
Observation 1: DISAGREES; reliability 0.75.
Observation 2: AGREES; reliability 0.75.
Observation 3: DISAGREES; reliability 0.75.

Which candidate has greater posterior probability after all observations?
Reply with exactly X or Y.
ANSWER:


In [16]:
reduced_metrics = MetricSpec(sequence_scores=True)  # X/Y resolve to probe values 2/7
reduced_capture = CaptureSpec(
    logits_boundaries=("answer",),
    streams=("resid_pre", "token_mixer_out", "mlp_out", "resid_post"),
    layers="all",
    tokens="all",
    every_decode_position=False,
)
raw_mock = TranscriptDataset(
    [
        {
            **row,
            "posterior_correct": row["answer_pattern_index"] % 3 != 0
            if row["ground_truth_choice"] is not None
            else None,
            "parse_compliance": True,
        }
        for row in raw_immediate
    ]
)
reduced_mock = TranscriptDataset(
    [
        {
            **row,
            "posterior_correct": True if row["ground_truth_choice"] is not None else None,
            "parse_compliance": True,
        }
        for row in reduced
    ]
)
summarize_representation_control(raw_mock, reduced_mock)

{'pair_count': 16,
 'eligible_pair_count': 12,
 'undefined_pair_count': 0,
 'forced_tie_pair_count': 4,
 'uniform': {'raw_accuracy': 0.6666666666666666,
  'reduced_accuracy': 1.0,
  'accuracy_delta': 0.33333333333333337,
  'raw_parse_compliance': 1.0,
  'reduced_parse_compliance': 1.0},
 'natural_distribution': {'raw_accuracy': 0.6875,
  'reduced_accuracy': 1.0,
  'accuracy_delta': 0.3125,
  'raw_parse_compliance': 1.0,
  'reduced_parse_compliance': 1.0},
 'transitions': {'both_correct': 8,
  'raw_only_correct': 0,
  'reduced_only_correct': 4,
  'neither_correct': 0},
 'rescue_rate': 1.0,
 'regression_rate': 0.0}

## Batched Qwen3.5 execution (explicit opt-in)

Replace `model_name_or_path` with a local or Hub checkpoint. The same runner supports dense 4B/9B checkpoints and the official 27B GPTQ-Int4 checkpoint through Transformers-compatible loading. Stage one is batched across all reasoning prompts before stage-two answer prompts are reconstructed and batched.

In [20]:
model_config = ModelConfig(model_name_or_path="Qwen/Qwen3.5-4B", dtype="auto")
execution = ExecutionConfig(
    experiment_dir=experiment_dir,
    run_id="qwen35_4b_batch2",
    batch_size=2,
    capture=CaptureSpec(
        logits_boundaries=("reasoning", "answer"),
        streams=("resid_pre", "token_mixer_out", "mlp_out", "resid_post"),
        layers=(0, -1),
        tokens="last",
    ),
)
RUN_MODEL = True
if RUN_MODEL:
    results = dataset.execute(QwenRunner(model_config), execution)
    print(results.summarize())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

{'row_count': 16, 'defined_row_count': 16, 'undefined_row_count': 0, 'undefined_uniform_fraction': 0.0, 'undefined_natural_mass': 0.0, 'forced_tie_count': 4, 'forced_tie_uniform_fraction': 0.25, 'forced_tie_natural_mass': 0.25, 'accuracy_eligible_count': 12, 'accuracy_scored_count': 12, 'uniform_defined_history_accuracy': 0.5, 'natural_distribution_accuracy': 0.5, 'natural_distribution_question_set_count': 2, 'uniform_history_parse_compliance': 0.9375, 'natural_distribution_parse_compliance': 0.9375}


In [22]:
if RUN_MODEL:
    first_result = results[0]
    run_dir = Path(experiment_dir) / "runs" / execution.run_id
    display({
        "qwen_prompt": first_result["answer_serialized_prompt"],
        "qwen_input_ids": first_result["answer_input_ids"],
        "qwen_input_tokens": first_result["answer_input_tokens"],
        "answer_surfaces": first_result["answer_surfaces"],
        "answer_surface_token_ids": first_result["answer_surface_token_ids"],
    })
    display(get_answer_surface_logits(first_result, run_dir))
    display(get_activation(
        first_result, run_dir, boundary="answer", stream="resid_post",
        layer=0, token_index=-1,
    ))

[{'schema_version': '1.0',
  'row_id': 'efee7fc4643ad49681e3d7df',
  'question_set_index': 0,
  'answer_pattern_index': 0,
  'answer_pattern': 'YYY',
  'domain': [1, 2, 3, 4, 5, 6, 7, 8],
  'n': 8,
  'k': 3,
  'reliabilities_exact': ['3/4', '3/4', '3/4'],
  'reliabilities': [0.75, 0.75, 0.75],
  'shared_reliability': True,
  'questions': [{'raw_draws': [5, 8, 6, 3], 'membership_set': [3, 5, 6, 8]},
   {'raw_draws': [1, 5, 8, 7], 'membership_set': [1, 5, 7, 8]},
   {'raw_draws': [6, 1, 5, 4], 'membership_set': [1, 4, 5, 6]}],
  'membership_sets': [[3, 5, 6, 8], [1, 5, 7, 8], [1, 4, 5, 6]],
  'observed_reports': ['YES', 'YES', 'YES'],
  'x': 2,
  'y': 7,
  'allow_same': False,
  'reasoning_budget': 40,
  'call_layout': 'conversation',
  'messages': [{'role': 'system',
    'content': 'Reason carefully from the stated noisy-channel model.'},
   {'role': 'user',
    'content': 'A value s is uniformly distributed over the integers 1 through 8.\nFor every question, the reported answer equals 

## Optional every-decode-position capture

This disk-heavy configuration is deliberately not executed here. It performs a batched teacher-forced pass after generation and stores tensors only at generated positions.

In [ ]:
every_decode_execution = ExecutionConfig(
    experiment_dir=experiment_dir,
    run_id="qwen35_every_decode_example",
    batch_size=2,
    capture=CaptureSpec(
        logits_boundaries=("answer",),
        streams=("resid_post",),
        layers=(-1,),
        every_decode_position=True,
    ),
)
every_decode_execution